In [14]:
# ==========================================
# VISUALIZATION SCRIPT
# ==========================================
import os
import re
import numpy as np
import pandas as pd
import open3d as o3d
import matplotlib.pyplot as plt

# 1. CONFIGURATION
EXPERIMENT = "training_second_square_flange"
FEATURE_NAME = "surface6"
WORKPIECE_NAME = "workpiece3"

WORKPIECE_PATH = "workpiece/workpiece3/workpiece.stl"
FEATURE_PATH = f"workpiece/workpiece3/{FEATURE_NAME}.stl"
POSES_PATH = f"viewpoints_candidate/testing_data/{EXPERIMENT}"
CSV_PATH = f"surface/{EXPERIMENT}/chamfer_results.csv"
AXIS_SIZE = 20.0
NUM_BINS = 4
DISTANCE_THRESHOLD = 1.0  # Threshold to remove overlapping points (mm)

# 2. HELPER FUNCTIONS
def make_arrow(direction='x', size=50.0, color=(0.6, 0.6, 0.6)):
    arrow = o3d.geometry.TriangleMesh.create_arrow(
        cylinder_radius=size * 0.05,
        cone_radius=size * 0.15,
        cylinder_height=size * 0.80,
        cone_height=size * 0.20,
        resolution=20,
    )
    if direction == 'x':
        R_align = arrow.get_rotation_matrix_from_xyz((0, np.pi / 2, 0))
        arrow.rotate(R_align, center=(0, 0, 0))
    arrow.paint_uniform_color(list(color))
    arrow.compute_vertex_normals()
    return arrow

def make_xz_arrows(transform=None, size=50.0, color=(0.6, 0.6, 0.6)):
    frame = make_arrow('x', size=size, color=color) + make_arrow('z', size=size, color=color)
    if transform is not None:
        frame.transform(transform)
    return frame

def visualize_viewpoints_colored(meshes, matrices, colors_per_frame, axis_size=50.0):
    if isinstance(matrices, np.ndarray) and matrices.ndim == 2:
        matrices = [matrices]

    geometries = list(meshes)
    geometries.append(make_xz_arrows(transform=None, size=axis_size, color=(0.9, 0.9, 0.9)))

    for i, (mat, color) in enumerate(zip(matrices, colors_per_frame)):
        geometries.append(make_xz_arrows(transform=mat, size=axis_size, color=color))

    print(f"\nVisualizing {len(matrices)} viewpoint(s) for feature '{FEATURE_NAME}'...")
    # If Open3D fails to show window in your environment, sometimes calling it without kwargs helps.
    o3d.visualization.draw_geometries(
        geometries,
        window_name="Feature Viewpoint Visualization",
        width=1024, height=768,
        front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
    )

def load_viewpoint_poses_dict(folder_path):
    def extract_number(filename):
        match = re.search(r'viewpoint_pose_(\d+)\.npy', filename)
        return int(match.group(1)) if match else -1
    
    npy_files = [f for f in os.listdir(folder_path) if f.endswith('.npy')]
    poses = {}
    for f in npy_files:
        idx = extract_number(f)
        if idx != -1:
            try:
                poses[idx] = np.load(os.path.join(folder_path, f))
            except Exception as e:
                print(f"Error loading {f}: {e}")
    return poses

# 3. LOAD DATA & COLOR PROCESSING
if os.path.exists(WORKPIECE_PATH):
    workpiece_mesh = o3d.io.read_triangle_mesh(WORKPIECE_PATH)
    workpiece_mesh.compute_vertex_normals()
    print("Sampling points from the whole workpiece mesh to create a PointCloud...")
    workpiece_pcd = workpiece_mesh.sample_points_poisson_disk(number_of_points=10000)
    workpiece_pcd.paint_uniform_color([0.6, 0.6, 0.6])  # Gray for the whole workpiece
else:
    print(f"Warning: {WORKPIECE_PATH} not found.")
    workpiece_pcd = o3d.geometry.PointCloud()

if os.path.exists(FEATURE_PATH):
    feature_mesh = o3d.io.read_triangle_mesh(FEATURE_PATH)
    feature_mesh.compute_vertex_normals()
    print("Sampling points from the feature mesh to create a PointCloud...")
    feature_pcd = feature_mesh.sample_points_poisson_disk(number_of_points=5000)
    feature_pcd.paint_uniform_color([1.0, 0.0, 0.0])  # Red for the feature
else:
    print(f"Warning: {FEATURE_PATH} not found.")
    feature_pcd = o3d.geometry.PointCloud()

# Remove overlapping points from the workpiece so the feature is perfectly clear
if len(workpiece_pcd.points) > 0 and len(feature_pcd.points) > 0:
    print("Removing points from the workpiece that overlap with the selected feature...")
    dists = workpiece_pcd.compute_point_cloud_distance(feature_pcd)
    dists = np.asarray(dists)
    non_feature_indices = np.where(dists > DISTANCE_THRESHOLD)[0]
    workpiece_pcd = workpiece_pcd.select_by_index(non_feature_indices)

poses_dict = load_viewpoint_poses_dict(POSES_PATH)
print(f"Loaded {len(poses_dict)} viewpoint poses from {POSES_PATH}")

df_chamfer = pd.read_csv(CSV_PATH)
df_feature = df_chamfer[df_chamfer['Feature'] == FEATURE_NAME].copy()
if df_feature.empty:
    print(f"No data found for feature '{FEATURE_NAME}' in CSV!")
else:
    print(f"Found {len(df_feature)} rows for feature '{FEATURE_NAME}'.")
    _cmap = plt.get_cmap('coolwarm')
    CLASS_COLORS = {cls: tuple(_cmap(cls / (NUM_BINS - 1))[:3]) for cls in range(NUM_BINS)}
    _, bins = pd.cut(df_feature['Chamfer_Distance_mm'], bins=NUM_BINS, retbins=True)
    df_feature['Error_Class'] = pd.cut(
        df_feature['Chamfer_Distance_mm'], bins=bins, labels=range(NUM_BINS), include_lowest=True
    )

valid_poses = []
valid_classes = []
for _, row in df_feature.iterrows():
    v_idx = int(row['Viewpoint'])
    if v_idx in poses_dict and pd.notna(row['Error_Class']):
        valid_poses.append(poses_dict[v_idx])
        valid_classes.append(int(row['Error_Class']))

colors_per_frame = [CLASS_COLORS[ec] for ec in valid_classes]
if not df_feature.empty:
    print(f"Chamfer Distance range: {df_feature['Chamfer_Distance_mm'].min():.2f} - {df_feature['Chamfer_Distance_mm'].max():.2f} mm")

# 4. SHOW RESULTS
if valid_poses:
    visualize_viewpoints_colored([workpiece_pcd, feature_pcd], valid_poses, colors_per_frame, axis_size=AXIS_SIZE)
else:
    print("No valid viewpoint poses to visualize.")


Sampling points from the whole workpiece mesh to create a PointCloud...
Sampling points from the feature mesh to create a PointCloud...
Removing points from the workpiece that overlap with the selected feature...
Loaded 432 viewpoint poses from viewpoints_candidate/testing_data/training_second_square_flange
Found 71 rows for feature 'surface6'.
Chamfer Distance range: -1.00 - 11.47 mm

Visualizing 71 viewpoint(s) for feature 'surface6'...
